# External data — bulk acquisition (v2)

Fixes the v1 misfires:
- **BiGG**: enumerate ALL bacterial models via the BiGG REST API (no more guessing IDs)
- **feba.db**: try multiple known sources; fall back to per-organism download if the SQLite is not bulk-hosted

Saves everything to **Drive → `cell_external_data/`** so the sandbox can pull individual files via the Drive connector (most BiGG models are <1 MB; feba.db is the only large file).

Runtime: CPU only, ~20 min.

In [ ]:
# 0. mount drive, deps
from google.colab import drive
drive.mount('/content/drive')
import os, json, csv, time
DST = '/content/drive/MyDrive/cell_external_data'
os.makedirs(f'{DST}/bigg_models', exist_ok=True)
os.makedirs(f'{DST}/feba',        exist_ok=True)
!pip -q install requests cobra >/dev/null 2>&1
import requests
print('OK ->', DST)

## Path 1 — BiGG: enumerate ALL bacterial models via API

In [ ]:
# 1a. enumerate every model in BiGG via the REST API
API = 'http://bigg.ucsd.edu/api/v2/models'
r = requests.get(API, timeout=60)
all_models = r.json()['results']
print(f'BiGG total models: {len(all_models)}')

# fetch per-model metadata (organism + taxonomy)
rich = []
for i, m in enumerate(all_models):
    try:
        d = requests.get(f"{API}/{m['bigg_id']}", timeout=30).json()
        rich.append(d)
        if i % 10 == 0: print(f'  meta {i}/{len(all_models)}: {m["bigg_id"]} ({d.get("organism","?")})')
        time.sleep(0.1)
    except Exception as e:
        print(f'  meta {m["bigg_id"]} fail: {e}')
print(f'\nfetched metadata for {len(rich)} models')
json.dump(rich, open(f'{DST}/bigg_all_models.json','w'), indent=2)

In [ ]:
# 1b. filter for bacteria (exclude yeast, human, mouse, hamster, etc.)
EUK_HINTS = ['sapiens','cerevisiae','musculus','cricetulus','danio','elegans',
             'human','yeast','mouse','rat','fly','arabidopsis','recon']
def is_bacterium(m):
    org = (m.get('organism') or '').lower()
    if any(e in org for e in EUK_HINTS): return False
    bid = m.get('bigg_id','').lower()
    if any(e in bid for e in EUK_HINTS): return False
    return True

bact = [m for m in rich if is_bacterium(m)]
print(f'bacterial models: {len(bact)} / {len(rich)}')
for m in bact[:20]:
    print(f"  {m['bigg_id']:<22} {m.get('organism','?')[:60]:<60} "
          f"({m.get('gene_count','?')} genes, {m.get('reaction_count','?')} rxns)")
print('  ...')

In [ ]:
# 1c. bulk-download every bacterial model. Try .xml.gz then .json then .xml.
import urllib.request
downloaded = []
failed = []
for m in bact:
    bid = m['bigg_id']
    out = None
    for ext in ('.xml.gz', '.json', '.xml', '.mat'):
        url = f'http://bigg.ucsd.edu/static/models/{bid}{ext}'
        path = f'{DST}/bigg_models/{bid}{ext}'
        if os.path.exists(path) and os.path.getsize(path) > 5_000:
            out = path; break
        try:
            req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
            data = urllib.request.urlopen(req, timeout=45).read()
            if len(data) > 5_000:
                open(path,'wb').write(data); out = path; break
        except Exception:
            continue
    if out:
        downloaded.append(dict(bigg_id=bid, organism=m.get('organism',''),
                                file=os.path.basename(out),
                                bytes=os.path.getsize(out)))
    else:
        failed.append(bid)
print(f'\nDOWNLOADED: {len(downloaded)} / {len(bact)} bacterial models')
print(f'failed: {failed[:10]}{" ..." if len(failed)>10 else ""}')

In [ ]:
# 1d. fuzzy-match BiGG models to our 59 organisms
OUR_ORGS = ['beril_Putida','beril_Keio','beril_BFirm','beril_Burk376',
  'beril_pseudo1_N1B4','beril_WCS417','beril_Cup4G11','beril_RalstoniaUW163',
  'beril_RalstoniaGMI1000','beril_pseudo5_N2C3_1','beril_azobra',
  'beril_RalstoniaBSBF1503','beril_Smeli','beril_Koxy','beril_HerbieS',
  'beril_SyringaeB728a','beril_Magneto','beril_pseudo6_N2E2',
  'beril_acidovorax_3H11','beril_SyringaeB728a_mexBdelta','beril_psRCH2',
  'beril_Dino','beril_RalstoniaPSI07','beril_pseudo13_GW456_L13','beril_PV4',
  'beril_PS','beril_Phaeo','beril_pseudo3_N2E3','beril_Marino',
  'beril_Ddia6719','beril_ANA3','beril_Korea','beril_Dda3937','mtub',
  'beril_SynE','beril_Dyella79','beril_Miya','beril_SB2B','beril_DdiaME23',
  'beril_MR1','beril_Ponti','beril_Caulo','beril_Cola','beril_Pedo557',
  'beril_DvH','beril_Keio','beril_Btheta','reut_full','beril_Kang',
  'beril_Methanococcus_S2','saur_NCTC8325_biotradis','aeromonas','syn3a',
  'mgen','beril_Methanococcus_JJ','reut','spneT4','spne19F',
  'ecoli_BW25113_tradis','mpne']

GENUS_HINTS = {
    'beril_Putida':'Pseudomonas putida', 'beril_Keio':'Escherichia coli',
    'ecoli_BW25113_tradis':'Escherichia coli',
    'beril_BFirm':'Burkholderia','beril_Burk376':'Burkholderia',
    'beril_pseudo1_N1B4':'Pseudomonas','beril_WCS417':'Pseudomonas',
    'beril_Cup4G11':'Cupriavidus','beril_RalstoniaUW163':'Ralstonia',
    'beril_RalstoniaGMI1000':'Ralstonia','beril_pseudo5_N2C3_1':'Pseudomonas',
    'beril_azobra':'Azospirillum','beril_RalstoniaBSBF1503':'Ralstonia',
    'beril_Smeli':'Sinorhizobium','beril_Koxy':'Klebsiella','beril_HerbieS':'Herbaspirillum',
    'beril_SyringaeB728a':'Pseudomonas syringae','beril_Magneto':'Magnetospirillum',
    'beril_pseudo6_N2E2':'Pseudomonas','beril_acidovorax_3H11':'Acidovorax',
    'beril_SyringaeB728a_mexBdelta':'Pseudomonas syringae','beril_psRCH2':'Pseudomonas',
    'beril_Dino':'Dinoroseobacter','beril_RalstoniaPSI07':'Ralstonia',
    'beril_pseudo13_GW456_L13':'Pseudomonas','beril_PV4':'Pseudovibrio',
    'beril_PS':'Pseudomonas','beril_Phaeo':'Phaeobacter','beril_pseudo3_N2E3':'Pseudomonas',
    'beril_Marino':'Marinobacter','beril_Ddia6719':'Dickeya','beril_ANA3':'Shewanella',
    'beril_Korea':'Korea','beril_Dda3937':'Dickeya','mtub':'Mycobacterium tuberculosis',
    'beril_SynE':'Synechococcus','beril_Dyella79':'Dyella','beril_Miya':'Miya',
    'beril_SB2B':'Shewanella','beril_DdiaME23':'Dickeya','beril_MR1':'Shewanella',
    'beril_Ponti':'Pontibacter','beril_Caulo':'Caulobacter','beril_Cola':'Colwellia',
    'beril_Pedo557':'Pedobacter','beril_DvH':'Desulfovibrio',
    'beril_Btheta':'Bacteroides thetaiotaomicron','reut_full':'Cupriavidus',
    'beril_Kang':'Kangiella','beril_Methanococcus_S2':'Methanococcus',
    'saur_NCTC8325_biotradis':'Staphylococcus aureus','aeromonas':'Aeromonas',
    'syn3a':'Mycoplasma','mgen':'Mycoplasma genitalium',
    'beril_Methanococcus_JJ':'Methanococcus','reut':'Cupriavidus',
    'spneT4':'Streptococcus pneumoniae','spne19F':'Streptococcus pneumoniae','mpne':'Mycoplasma pneumoniae',
}

manifest = []
for org in OUR_ORGS:
    hint = GENUS_HINTS.get(org,'').lower()
    if not hint: continue
    matches = [d for d in downloaded if hint.split()[0] in d['organism'].lower()]
    if hint and ' ' in hint:
        species = hint.split()[1]
        spec = [d for d in matches if species in d['organism'].lower()]
        if spec: matches = spec
    for m in matches:
        manifest.append(dict(our_organism=org, bigg_id=m['bigg_id'],
                              organism=m['organism'], file_basename=m['file']))

# write manifest
with open(f'{DST}/bigg_manifest.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['our_organism','bigg_id','organism','file_basename'])
    for r in manifest: w.writerow([r['our_organism'], r['bigg_id'], r['organism'], r['file_basename']])
matched = sorted(set(m['our_organism'] for m in manifest))
print(f'\nOur organisms with a BiGG-model match: {len(matched)} / {len(OUR_ORGS)}')
for o in matched: print(f'  {o}')
print(f'\nfull manifest rows: {len(manifest)}')

In [ ]:
# 1e. validate every downloaded model loads + computes growth
import cobra, cobra.io
from cobra import Configuration
Configuration().solver = 'glpk'
validation = []
for d in downloaded:
    p = f'{DST}/bigg_models/{d["file"]}'
    try:
        if p.endswith('.json'):
            m = cobra.io.load_json_model(p)
        elif p.endswith('.mat'):
            m = cobra.io.load_matlab_model(p)
        else:
            m = cobra.io.read_sbml_model(p)
        g = m.slim_optimize()
        validation.append(dict(bigg_id=d['bigg_id'], rxns=len(m.reactions),
                                mets=len(m.metabolites), genes=len(m.genes),
                                growth=round(float(g) if g else 0, 4)))
    except Exception as e:
        validation.append(dict(bigg_id=d['bigg_id'], error=str(e)[:80]))
json.dump(validation, open(f'{DST}/bigg_validation.json','w'), indent=2)
ok = [v for v in validation if 'growth' in v]
print(f'{len(ok)} / {len(downloaded)} models load + grow OK')
for v in ok[:15]:
    print(f'  {v["bigg_id"]:<22} {v["rxns"]:>5} rxns | {v["genes"]:>5} genes | growth={v["growth"]}')

## Path 2 — Fitness Browser feba.db (try multiple known sources)

In [ ]:
# 2a. try multiple known feba.db sources in order. The Fitness Browser
# (Morgan Price, Arkin lab) ships its SQLite at varying paths over the years.
FEBA_URLS = [
    'https://fit.genomics.lbl.gov/cgi_data/feba.db',
    'http://fit.genomics.lbl.gov/cgi_data/feba.db',
    'https://genomics.lbl.gov/supplemental/bigfit/feba.db',
    'http://genomics.lbl.gov/supplemental/bigfit/feba.db',
    'https://fit.genomics.lbl.gov/data/feba.db',
    'https://fit.genomics.lbl.gov/static/feba.db',
]
feba_path = f'{DST}/feba/feba.db'
got_feba = False
if os.path.exists(feba_path) and os.path.getsize(feba_path) > 1e8:
    print(f'feba.db already cached ({os.path.getsize(feba_path)/1e9:.1f} GB)')
    got_feba = True
else:
    for url in FEBA_URLS:
        print(f'try  {url}')
        rc = os.system(f'wget -q --timeout=60 -O "{feba_path}.tmp" "{url}"')
        sz = os.path.getsize(f'{feba_path}.tmp') if os.path.exists(f'{feba_path}.tmp') else 0
        if rc == 0 and sz > 1e8:
            os.rename(f'{feba_path}.tmp', feba_path)
            print(f'  OK -- {sz/1e9:.2f} GB'); got_feba = True; break
        else:
            os.system(f'rm -f "{feba_path}.tmp"')
            print(f'  fail ({sz} bytes)')
print(f'\nfeba.db acquired: {got_feba}')

In [ ]:
# 2b. fallback if feba.db direct download didn't work: clone the morgannprice/feba
#     github repo (only the data subdirs we need) and look for fitness summary TSVs.
if not got_feba:
    print('fallback: cloning morgannprice/feba (depth 1) ...')
    if not os.path.exists('/content/feba_repo'):
        rc = os.system('git clone --depth 1 https://github.com/morgannprice/feba /content/feba_repo 2>&1')
        print(f'  clone rc={rc}')
    if os.path.exists('/content/feba_repo'):
        print('  repo files (top-level):')
        for f in sorted(os.listdir('/content/feba_repo'))[:30]: print(f'    {f}')
        # look for any sqlite
        for root, dirs, files in os.walk('/content/feba_repo'):
            for f in files:
                if f.endswith('.db') or f.endswith('.sqlite') or 'feba' in f.lower():
                    full = os.path.join(root, f)
                    print(f'  candidate: {full}  ({os.path.getsize(full)/1e6:.1f} MB)')
        # look for data subdirectories
        print('\n  subdirectories with data hints:')
        for d in os.listdir('/content/feba_repo'):
            full = os.path.join('/content/feba_repo', d)
            if os.path.isdir(full):
                child = sorted(os.listdir(full))[:5]
                print(f'    {d}/ : {child}')

In [ ]:
# 2c. last resort: scrape the Fitness Browser per-organism gene+fitness tables
# from the public CGI (one organism at a time). Slow but reliable.
if not got_feba:
    # discover org IDs from the front page
    front = requests.get('https://fit.genomics.lbl.gov/cgi-bin/myFrontPage.cgi',
                          timeout=30, headers={'User-Agent':'Mozilla/5.0'}).text
    import re
    org_ids = sorted(set(re.findall(r'orgId=([A-Za-z0-9_]+)', front)))
    print(f'discovered {len(org_ids)} organism IDs on Fitness Browser')
    print('sample:', org_ids[:20])
    # save the list -- the sandbox can decide which orgs to pull per-condition
    open(f'{DST}/feba/orgIds.txt','w').write('\n'.join(org_ids))
    print(f'wrote orgIds.txt ({len(org_ids)} orgs)')

In [ ]:
# 3. final inventory
print('=== Drive cell_external_data/ ===')
total = 0
for root, dirs, files in os.walk(DST):
    for f in files:
        p = os.path.join(root, f); s = os.path.getsize(p); total += s
        if s > 10_000 or f.endswith(('.csv','.json','.txt','.md')):
            print(f'  {s/1024:>10.1f} KB  {p.replace(DST+"/","")}')
print(f'\nTotal: {total/1e9:.2f} GB')
print(f'\nBiGG models downloaded: {len(downloaded)}')
print(f'BiGG models matched to our organisms: {len(set(m["our_organism"] for m in manifest))}/{len(OUR_ORGS)}')
print(f'feba.db acquired: {got_feba}')